In [1]:
import json
import numpy as np
from PIL import Image

def load_annotation(file_path):
    data = []

    with open (file_path, 'r', encoding="utf-8") as file:
        data = json.load(file)    
    return data

annotation_path = "data/vimmsd-public-test.json"
data = load_annotation(annotation_path)
data

{'0': {'image': 'bb934d7d7f7652903c24272d405e4b31c70689cec93f86d5d36f17707c36fceb.jpg',
  'caption': 'Hãy thực tế lên, đừng kêu ca nữa :)))',
  'label': None},
 '1': {'image': '449a108232be3d220679d7c500ee0aa3203920c01163190bf900456af85564f6.jpg',
  'caption': 'Thôi thế cho anh mua gói lạc',
  'label': None},
 '2': {'image': '2c17cee0245711376b566a65d1a9d25066317fc894fc0a7a98543dada5883e1b.jpg',
  'caption': 'Chính quyền bang và đơn vị vận hành lưới điện đang vật lộn với tình thế kỳ quặc. Quá nhiều điện mặt trời vào những ngày nắng khi nhu cầu không quá cao dẫn tới giá điện âm\nĐiện mặt trời có nhiều lợi ích tuyệt vời: gần như không tốn chi phí vận hành sau khi xây dựng, không tạo ra ô nhiễm không khí và sản xuất năng lượng mà không cần đốt nhiên liệu hóa thạch. Nhưng một bất lợi chính của nó là Mặt Trời không chiếu sáng mọi lúc.\nCách đây hơn 15 năm, nhóm nghiên cứu ở Phòng thí nghiệm năng lượng tái tạo quốc gia lập mô hình điện mặt trời phổ biến trong tương lai và nhận ra điều kỳ lạ.

In [2]:
import pandas as pd
import numpy as np
import clip
import torch
import tqdm
import json
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:


# Set the device to GPU if available, otherwise to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Define the two labels for classification
labels = ["sarcasm", "not-sarcasm"]

# Preprocess the text descriptions for each label
text_inputs = processor(text=[f"a photo of {label}" for label in labels], return_tensors="pt", padding=True).to(device)

# Load dataset
dataset = pd.read_csv(r'/content/train-sarcasm-dataset.csv', encoding='utf-8-sig')

# Select indices for three example images
indices = [0, 23, 10]

# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Loop through the indices and process each image
for i, idx in enumerate(indices):
    example = dataset.iloc[idx]  # Sử dụng iloc để truy cập hàng theo chỉ số
    image_path = f"/content/drive/MyDrive/train-images/train-images/{example['Image']}"  # Lấy tên ảnh từ cột 'image'
    image = Image.open(image_path).convert("RGB")
    label = example['Label']  # Lấy nhãn từ cột 'label'

    # Preprocess the image
    image_input = processor(images=image, return_tensors="pt").to(device)

    # Calculate image and text features
    with torch.no_grad():
        outputs = model(**image_input, input_ids=text_inputs.input_ids)
        image_features = outputs.image_embeds
        text_features = outputs.text_embeds

    # Normalize the features
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Calculate similarity between image and text features
    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    values, predicted_indices = similarity[0].topk(1)

    # Display the image in the subplot
    axes[i].imshow(image)
    axes[i].set_title(f"Predicted: {labels[predicted_indices[0]]}, Actual: {label}")
    axes[i].axis('off')

# Show the plot
plt.tight_layout()
plt.show()
